# 01 — Data setup: CUAD vendor service agreements (maintenance & services)

**What this notebook does (run top to bottom, ~1 minute):**
1. Loads CUAD from your Zenodo `CUAD_v1.zip` if you uploaded it, otherwise downloads the same data from GitHub.
2. Selects our **15 vendor service agreements**: one contract type, where a provider serves a paying business customer (maintenance is a subset).
3. Saves each contract as a `.txt` file so we can read and label it.
4. Extracts CUAD's highlighted spans for the clause families CUAD covers → the answer key for **Track A (clause detection)**.
5. Splits contracts into **dev (4)** and **test (11)**.
6. Creates one labeling spreadsheet per annotator (Kayla, Jenny) with dropdowns → becomes the answer key for **Track B (risk verdicts)** after we compare and adjudicate.

Everything is written to `cuad_project_data/`. The last cell zips it for download.

In [ ]:
import os, json, re, random, zipfile, glob
import pandas as pd

OUT = "cuad_project_data"
os.makedirs(f"{OUT}/contracts", exist_ok=True)

# 1) Upload CUAD from your computer (CUAD_v1.zip, or CUAD_v1.json if you unzipped it)
from google.colab import files
uploaded = files.upload()                      # waits until the upload is 100% done
name = list(uploaded)[0]
print("Uploaded:", name, round(len(uploaded[name]) / 1e6, 1), "MB")

if name.lower().endswith(".zip"):
    if not zipfile.is_zipfile(name):
        raise ValueError("This zip is broken or incomplete. Re-download CUAD_v1.zip "
                         "from Zenodo (about 106 MB) and run this cell again.")
    zipfile.ZipFile(name).extractall("cuad_raw")
    json_path = [f for f in glob.glob("cuad_raw/**/CUAD_v1.json", recursive=True)
                 if "__MACOSX" not in f][0]
else:
    json_path = name                           # you uploaded CUAD_v1.json directly

cuad = json.load(open(json_path))["data"]
print(len(cuad), "contracts in CUAD")

## 2. Our 15 contracts

**Contract type: vendor service agreements.** A provider performs ongoing services (maintenance, support, IT, outsourcing, professional services) for a business customer that pays for them. Out of scope: transportation, marketing/promotion partnerships, financing "capital maintenance" agreements, employment, affiliate deals.

We started from the 68 CUAD contracts titled maintenance or service agreements and removed fund/insurance paperwork and amendments, fragments under 1,000 words, contracts over 8,000 words (9,000 for C15, see below), and ones that aren't a provider serving a customer (employment deals, agreements between affiliates). Several contain cross-references ("see Section 9"), which we need for the demo.

`ALTERNATES` are swap-ins if a contract turns out to be unusable (e.g. heavily redacted). They are also service agreements.

In [ ]:
SELECTED = {
    # id    : (short name,                         CUAD title)
    "C01": ("Garman software maintenance",        "SPARKLINGSPRINGWATERHOLDINGSLTD_07_03_2002-EX-10.13-SOFTWARE LICENSE AND MAINTENANCE AGREEMENT"),
    "C02": ("Kubient master services",            "KUBIENT,INC_07_02_2020-EX-10.14-MASTER SERVICES AGREEMENT_Part1"),
    "C03": ("Medical Manager IT maintenance",     "TRIZETTOGROUPINC_08_18_1999-EX-10.17-TECHNICAL INFRASTRUCTURE MAINTENANCE AGREEMENT"),
    "C04": ("CityNet network maintenance",        "UAGHINC_04_14_2004-EX-10.18-MAINTENANCE AGREEMENT"),
    "C05": ("CRO Consulting master services",     "PAXMEDICA,INC_07_02_2020-EX-10.12-Master Service Agreement"),
    "C06": ("RadialSpark master services",        "MERITLIFEINSURANCECO_06_19_2020-EX-10.(XIV)-MASTER SERVICES AGREEMENT"),
    "C07": ("XIMAGE system maintenance",          "IMAGEWARESYSTEMSINC_12_20_1999-EX-10.22-MAINTENANCE AGREEMENT"),
    "C08": ("Sicap software maintenance/support", "SLOVAKWIRELESSFINANCECOBV_03_28_2001-EX-4.(B)(II).3-Maintenance and support contract for SICAP(R) modules"),
    "C09": ("Leader software maintenance",        "HerImports_20161018_8-KA_EX-10.14_9765707_EX-10.14_Maintenance Agreement"),
    "C10": ("E-Path software maintenance",        "SFGFINANCIALCORP_05_12_2009-EX-10.1-SOFTWARE LICENSE AND MAINTENANCE AGREEMENT"),
    "C11": ("Mobimagic service agreement",        "OPERALTD_04_30_2020-EX-4.14-SERVICE AGREEMENT"),
    "C12": ("Telcostar services agreement",       "ABILITYINC_06_15_2020-EX-4.25-SERVICES AGREEMENT"),
    "C13": ("StartEngine services agreement",     "SOLUTIONSVENDINGINTERNATIONAL,INC_03_31_2020-EX1A-1 UNDR AGMT-SERVICES AGREEMENT"),
    "C14": ("XACCT support and maintenance",      "XACCT Technologies, Inc.SUPPORT AND MAINTENANCE AGREEMENT"),
    "C15": ("On2/Wildform support and maintenance", "ON2TECHNOLOGIES,INC_11_17_2006-EX-10.3-SUPPORT AND MAINTENANCE AGREEMENT"),
    # C15 was the Dynamex transportation agreement until 2026-10-21: transportation is out of scope.
    # All 15 now come from CUAD's Maintenance or Service folders. C15 is 8,844 words (cap raised to 9,000 for it).
}
ALTERNATES = [   # CUAD Maintenance folder, in scope, 1k-9k words, independent parties
    "VERTEXENERGYINC_08_14_2014-EX-10.24-OPERATION AND MAINTENANCE AGREEMENT",            # 8,601 words
    "SPIENERGYCO,LTD_03_09_2011-EX-99.5-OPERATIONS AND MAINTENANCE AGREEMENT",            # 8,153 words
]

def find(sub):
    hits = [x for x in cuad if x["title"].strip() == sub.strip()]
    if not hits:
        hits = [x for x in cuad if sub.strip().lower() in x["title"].lower()]
    assert len(hits) == 1, f"{sub}: {len(hits)} matches"
    return hits[0]

docs = {cid: find(sub) for cid, (_, sub) in SELECTED.items()}
print("Found all", len(docs), "contracts")

## 3. Save contract texts + a contracts table with the dev/test split

In [ ]:
random.seed(42)
dev_ids = sorted(random.sample(sorted(docs), 4))   # 4 dev for prompt tuning; never tune on the other 11

rows = []
for cid, x in docs.items():
    text = x["paragraphs"][0]["context"]
    with open(f"{OUT}/contracts/{cid}.txt", "w", encoding="utf-8") as f:
        f.write(text)
    rows.append(dict(contract_id=cid, short_name=SELECTED[cid][0], cuad_title=x["title"],
                     words=len(text.split()), split="dev" if cid in dev_ids else "test"))
contracts = pd.DataFrame(rows)
contracts.to_csv(f"{OUT}/contracts.csv", index=False)
print("Dev:", dev_ids, "| total words to read:", contracts.words.sum())
contracts

## 4. CUAD answer key for clause detection (Track A)

CUAD only covers part of our six families. **Primary** CUAD categories are scored as the Track A answer key. **Context** categories are related but not the same thing (e.g. Minimum Commitment is not payment terms), so they only appear as hints for annotators. For payment and confidentiality there is no CUAD category at all, so our own evidence spans are the answer key. `cuad_gap_table.csv` lists what the rubric asks about that CUAD cannot score. This is the gap we name in the report.


In [ ]:
# ==== CUAD -> our 6 families: mapping + coverage/gap tables ====
# Needs `cuad`, `docs` from the cells above. Writes 4 CSVs into cuad_project_data/.

# Primary categories = scored as the Track A answer key.
CUAD_TO_FAMILY = {
    "Renewal Term":                        "renewal",
    "Notice Period To Terminate Renewal":  "renewal",
    "Termination For Convenience":         "termination",
    "Cap On Liability":                    "liability_indemnity",
    "Uncapped Liability":                  "liability_indemnity",
    "Governing Law":                       "dispute_governing_law",
}
# Context categories = related but NOT the same thing. Shown as hints to annotators, never scored.
CUAD_CONTEXT = {
    "Expiration Date":           "renewal",
    "Post-Termination Services": "termination",
    "Change Of Control":         "termination",
    "Liquidated Damages":        "liability_indemnity",
    "Insurance":                 "liability_indemnity",
    "Warranty Duration":         "liability_indemnity",
    "Covenant Not To Sue":       "liability_indemnity",
    "Minimum Commitment":        "payment",
    "Price Restrictions":        "payment",
    "Most Favored Nation":       "payment",
    "Audit Rights":              "payment",
    "Revenue/Profit Sharing":    "payment",
}
FAMILIES = ["payment", "termination", "liability_indemnity", "confidentiality", "renewal", "dispute_governing_law"]

# What rubric_v1.md asks about that CUAD has no category for.
RUBRIC_NEEDS_NOT_IN_CUAD = {
    "payment":               "payment window, late fees, disputed invoices, suspension, price increases",
    "termination":           "termination for cause, cure period, early-termination fees, refunds",
    "liability_indemnity":   "indemnification (both directions), consequential-damages exclusion, carve-outs",
    "confidentiality":       "everything: no confidentiality category in CUAD",
    "renewal":               "renewal pricing",
    "dispute_governing_law": "venue, arbitration, jury waiver, attorney fees",
}
NOT_IN_CUAD = {"payment": "no CUAD category", "confidentiality": "no CUAD category",
               "termination": "partial: for-convenience only, no for-cause/cure period",
               "liability_indemnity": "partial: caps only, no indemnification",
               "dispute_governing_law": "partial: governing law only, no arbitration/venue"}

# 1) Check every name against the real CUAD file (names have odd punctuation; a typo = silent zero rows)
all_cats = sorted({qa["id"].split("__")[-1] for x in cuad for qa in x["paragraphs"][0]["qas"]})
print(len(all_cats), "CUAD categories found (expect 41)")
typos = [c for c in list(CUAD_TO_FAMILY) + list(CUAD_CONTEXT) if c not in all_cats]
assert not typos, f"Not real CUAD category names, fix them: {typos}"

# 2) Pull spans for our 15 contracts
spans = []
for cid, x in docs.items():
    for qa in x["paragraphs"][0]["qas"]:
        cat = qa["id"].split("__")[-1]
        role = "primary" if cat in CUAD_TO_FAMILY else "context" if cat in CUAD_CONTEXT else None
        if role:
            fam = CUAD_TO_FAMILY.get(cat) or CUAD_CONTEXT[cat]
            for a in qa["answers"]:
                spans.append(dict(contract_id=cid, family=fam, cuad_category=cat, role=role,
                                  answer_start=a["answer_start"], text=a["text"]))
all_spans = pd.DataFrame(spans, columns=["contract_id", "family", "cuad_category", "role",
                                         "answer_start", "text"]).drop_duplicates()
cuad_spans = all_spans[all_spans.role == "primary"].drop(columns="role")   # Track A answer key
cuad_context_spans = all_spans[all_spans.role == "context"].drop(columns="role")
cuad_spans.to_csv(f"{OUT}/cuad_gold_spans.csv", index=False)
cuad_context_spans.to_csv(f"{OUT}/cuad_context_spans.csv", index=False)
print(len(cuad_spans), "primary spans,", len(cuad_context_spans), "context spans across our 15 contracts")

# 3) Category table: how many of our 15 contracts CUAD actually labeled, per category
cat_rows = []
for cat, fam, role in ([(c, f, "primary") for c, f in CUAD_TO_FAMILY.items()] +
                       [(c, f, "context") for c, f in CUAD_CONTEXT.items()]):
    s = all_spans[all_spans.cuad_category == cat]
    cat_rows.append(dict(family=fam, cuad_category=cat, role=role,
                         contracts_labeled=s.contract_id.nunique(), n_spans=len(s)))
cat_table = pd.DataFrame(cat_rows).sort_values(["family", "role", "cuad_category"])
cat_table.to_csv(f"{OUT}/cuad_category_table.csv", index=False)

# 4) Family gap table: can CUAD be the answer key for this family?
gap_rows = []
for fam in FAMILIES:
    prim = [c for c, f in CUAD_TO_FAMILY.items() if f == fam]
    ctx = [c for c, f in CUAD_CONTEXT.items() if f == fam]
    n = cuad_spans[cuad_spans.family == fam].contract_id.nunique()
    status = "NO CUAD CATEGORY" if not prim else ("partial" if fam in NOT_IN_CUAD else "covered")
    gap_rows.append(dict(
        family=fam, cuad_status=status,
        primary_categories=", ".join(prim) or "-",
        context_categories=", ".join(ctx) or "-",
        contracts_with_cuad_span=f"{n}/{len(docs)}",
        not_in_cuad=RUBRIC_NEEDS_NOT_IN_CUAD[fam],
        track_a_answer_key="our evidence spans" if not prim else "CUAD spans + our spans for the gaps"))
gap_table = pd.DataFrame(gap_rows)
gap_table.to_csv(f"{OUT}/cuad_gap_table.csv", index=False)

# 5) Per-contract coverage (same file name as before so later cells still work)
cov = (cuad_spans.groupby(["contract_id", "family"]).size().unstack(fill_value=0) > 0)
cov = cov.reindex(index=sorted(docs), columns=FAMILIES, fill_value=False)
cov = cov.astype(object).replace({True: "CUAD span", False: "none"})
for fam, note in NOT_IN_CUAD.items():
    if note == "no CUAD category":
        cov[fam] = note
cov.to_csv(f"{OUT}/cuad_coverage.csv")

display(gap_table)
display(cat_table)
cov


## 5. Labeling spreadsheets (one per annotator)

**Rules: label on your own, don't look at the other person's sheet until both are done.** 15 contracts × 6 families = 90 rows each.

- `verdict`: acceptable / risky / missing / unclear (decided by the rubric)
- `severity`: high / medium / low / n/a
- `evidence_text`: paste the exact contract text; `evidence_2` for a second location (e.g. a notice period defined in another section)
- `cuad_hint`: CUAD's highlight, only to help you *find* the clause. You still decide the verdict.

In [ ]:
from openpyxl import Workbook
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.styles import Font, Alignment, PatternFill

def hint(cid, fam):
    s = cuad_spans[(cuad_spans.contract_id == cid) & (cuad_spans.family == fam)]
    c = cuad_context_spans[(cuad_context_spans.contract_id == cid) & (cuad_context_spans.family == fam)]
    parts = [f"[{r.cuad_category}] {r.text[:200]}" for r in s.itertuples()]
    parts += [f"(context) [{r.cuad_category}] {r.text[:150]}" for r in c.itertuples()]
    if not parts:
        return NOT_IN_CUAD.get(fam, "CUAD has no highlight here")
    return " || ".join(parts)

COLS = ["contract_id", "short_name", "split", "family", "verdict", "severity",
        "evidence_text", "section_ref", "evidence_2", "section_ref_2", "notes", "cuad_hint"]
WIDTHS = [11, 30, 7, 22, 13, 10, 60, 11, 45, 11, 40, 60]

def make_sheet(annotator):
    wb = Workbook(); ws = wb.active; ws.title = "labels"
    ws.append(COLS)
    for c in ws[1]:
        c.font = Font(bold=True); c.fill = PatternFill("solid", fgColor="DDE7F0")
    for r in contracts.itertuples():
        for fam in FAMILIES:
            ws.append([r.contract_id, r.short_name, r.split, fam, "", "", "", "", "", "", "", hint(r.contract_id, fam)])
    n = ws.max_row
    for col, opts in (("E", "acceptable,risky,missing,unclear"), ("F", "high,medium,low,n/a")):
        dv = DataValidation(type="list", formula1=f'"{opts}"', allow_blank=True)
        ws.add_data_validation(dv); dv.add(f"{col}2:{col}{n}")
    for i, w in enumerate(WIDTHS):
        ws.column_dimensions[chr(65 + i)].width = w
    for row in ws.iter_rows(min_row=2):
        for c in row:
            c.alignment = Alignment(wrap_text=True, vertical="top")
    ws.freeze_panes = "E2"
    path = f"{OUT}/labels_{annotator}.xlsx"
    wb.save(path); return path

for who in ["kayla", "jenny"]:
    print("wrote", make_sheet(who))

## 6. Zip everything for download / upload to the GitHub repo

In [ ]:
shutil_zip = f"{OUT}.zip"
with zipfile.ZipFile(shutil_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for fn in files:
            z.write(os.path.join(root, fn))
print("wrote", shutil_zip)
try:
    from google.colab import files
    files.download(shutil_zip)
except ImportError:
    pass